# 🤖 Panduan Lengkap Supervised Machine Learning
### Dari Data Mentah hingga Model Siap Pakai

---

Notebook ini memandu kamu melalui **9 tahap utama** dalam supervised learning:

| # | Tahap | Isi |
|---|-------|-----|
| 1 | **Setup & Install** | Library yang dibutuhkan |
| 2 | **Load & Eksplorasi Data (EDA)** | Memahami data sebelum diproses |
| 3 | **Data Preprocessing** | Bersihkan & siapkan data |
| 4 | **Feature Engineering** | Buat & pilih fitur terbaik |
| 5 | **Train/Test Split** | Bagi data secara benar |
| 6 | **Training Banyak Algoritma** | Latih & bandingkan 9 model |
| 7 | **Evaluasi Model** | Ukur performa dengan metrik yang tepat |
| 8 | **Hyperparameter Tuning** | Optimasi model terbaik |
| 9 | **Simpan & Load Model** | Persiapan deployment |

> 💡 **Dataset yang digunakan:** Iris (klasifikasi) dan Diabetes (regresi) dari scikit-learn — sudah tersedia tanpa perlu download.

---
## 📦 Tahap 1: Setup & Install Library

In [ ]:
# ============================================================
# INSTALL LIBRARY TAMBAHAN
# (scikit-learn, pandas, matplotlib sudah ada di Colab)
# xgboost dan imbalanced-learn perlu diinstall manual
# ============================================================
!pip install xgboost imbalanced-learn --quiet

print("✅ Instalasi selesai!")

In [ ]:
# ============================================================
# IMPORT SEMUA LIBRARY
# Kita import semua di awal agar tidak perlu import ulang
# ============================================================

# --- Library dasar ---
import numpy as np              # operasi numerik dan array
import pandas as pd             # manipulasi data berbentuk tabel (DataFrame)
import warnings
warnings.filterwarnings('ignore')  # sembunyikan warning yang tidak penting

# --- Visualisasi ---
import matplotlib.pyplot as plt    # grafik dasar
import seaborn as sns              # grafik statistik yang lebih cantik
plt.rcParams['figure.figsize'] = (10, 6)  # ukuran default grafik
plt.rcParams['font.size'] = 11

# --- Dataset bawaan scikit-learn ---
from sklearn.datasets import load_iris, load_breast_cancer, load_diabetes

# --- Preprocessing ---
from sklearn.model_selection import (
    train_test_split,       # bagi data train/test
    cross_val_score,        # evaluasi dengan cross-validation
    GridSearchCV,           # cari hyperparameter terbaik (exhaustive)
    RandomizedSearchCV      # cari hyperparameter terbaik (acak, lebih cepat)
)
from sklearn.preprocessing import (
    StandardScaler,         # normalisasi: mean=0, std=1
    MinMaxScaler,           # normalisasi ke rentang [0, 1]
    LabelEncoder,           # encode label kategorikal ke angka
    OneHotEncoder           # encode kategorikal ke kolom biner
)
from sklearn.impute import SimpleImputer   # isi nilai yang hilang (missing values)
from sklearn.pipeline import Pipeline      # rangkai preprocessing + model jadi satu alur

# --- Algoritma Klasifikasi ---
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier
)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

# --- Algoritma Regresi ---
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor

# --- Metrik Evaluasi ---
from sklearn.metrics import (
    # Klasifikasi
    accuracy_score,         # akurasi keseluruhan
    precision_score,        # dari prediksi positif, berapa yang benar?
    recall_score,           # dari positif sejati, berapa yang terdeteksi?
    f1_score,               # harmonic mean dari precision dan recall
    roc_auc_score,          # area under ROC curve
    classification_report,  # laporan lengkap per kelas
    confusion_matrix,       # tabel prediksi benar vs salah
    ConfusionMatrixDisplay,
    # Regresi
    r2_score,               # proporsi variansi yang dijelaskan model
    mean_absolute_error,    # rata-rata error absolut
    mean_squared_error,     # rata-rata error kuadrat
)

# --- Lainnya ---
import joblib   # simpan dan load model
import time     # hitung waktu training

print("✅ Semua library berhasil diimport!")

---
## 🔍 Tahap 2: Load Data & Exploratory Data Analysis (EDA)

EDA adalah proses **memahami data** sebelum diproses. Tujuannya:
- Mengetahui bentuk dan tipe data
- Menemukan nilai yang hilang (missing values)
- Melihat distribusi dan outlier
- Memahami hubungan antar fitur

In [ ]:
# ============================================================
# LOAD DATASET
# Kita gunakan dataset Iris untuk KLASIFIKASI
# Target: prediksi spesies bunga (3 kelas)
# ============================================================

# Load dataset dari scikit-learn
iris = load_iris()

# Ubah ke DataFrame pandas agar lebih mudah dianalisis
df = pd.DataFrame(
    data=iris.data,             # fitur (input)
    columns=iris.feature_names  # nama kolom
)

# Tambahkan kolom target (label yang ingin diprediksi)
df['target'] = iris.target
df['species'] = df['target'].map({
    0: 'setosa',
    1: 'versicolor',
    2: 'virginica'
})

print(f"Dataset: Iris")
print(f"Jumlah baris (sampel): {df.shape[0]}")
print(f"Jumlah kolom (fitur + target): {df.shape[1]}")
print(f"Kelas target: {iris.target_names.tolist()}")

In [ ]:
# ============================================================
# LIHAT SEKILAS DATA
# ============================================================

# .head() menampilkan 5 baris pertama
# Gunakan ini untuk memastikan data terloading dengan benar
print("=== 5 Baris Pertama ===")
df.head()

In [ ]:
# ============================================================
# STATISTIK DESKRIPTIF
# Ringkasan statistik: mean, std, min, max, kuartil
# Berguna untuk melihat skala data dan outlier kasar
# ============================================================
print("=== Statistik Deskriptif ===")
df.describe().round(2)

In [ ]:
# ============================================================
# CEK TIPE DATA DAN MISSING VALUES
# Tipe data penting: int/float = numerik, object = kategorikal
# Missing values (NaN) harus ditangani sebelum training
# ============================================================
print("=== Tipe Data ===")
print(df.dtypes)
print()

print("=== Missing Values per Kolom ===")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    'Jumlah Missing': missing,
    'Persentase (%)': missing_pct
})
print(missing_df)
print()

# Cek duplikat (baris yang persis sama)
print(f"=== Duplikat ===")
print(f"Jumlah baris duplikat: {df.duplicated().sum()}")

In [ ]:
# ============================================================
# DISTRIBUSI KELAS TARGET
# Penting untuk klasifikasi: apakah kelas seimbang?
# Jika sangat tidak seimbang (imbalanced), perlu penanganan khusus
# ============================================================
print("=== Distribusi Kelas Target ===")
print(df['species'].value_counts())
print()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart distribusi kelas
df['species'].value_counts().plot(kind='bar', ax=axes[0], color=['#185FA5','#1D9E75','#D85A30'])
axes[0].set_title('Distribusi Kelas Target')
axes[0].set_xlabel('Spesies')
axes[0].set_ylabel('Jumlah Sampel')
axes[0].tick_params(rotation=0)

# Pie chart
df['species'].value_counts().plot(kind='pie', ax=axes[1],
    colors=['#185FA5','#1D9E75','#D85A30'], autopct='%1.1f%%')
axes[1].set_title('Proporsi Kelas')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

# Dataset ini BALANCED (seimbang) — 50 sampel per kelas
# Kalau tidak seimbang, perlu teknik seperti SMOTE atau class_weight

In [ ]:
# ============================================================
# DISTRIBUSI TIAP FITUR
# Histogram menunjukkan sebaran nilai untuk tiap fitur
# Boxplot menunjukkan median, kuartil, dan outlier
# ============================================================
fitur_numerik = iris.feature_names

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i, col in enumerate(fitur_numerik):
    # Histogram — baris pertama
    axes[0, i].hist(df[col], bins=20, color='#185FA5', edgecolor='white', alpha=0.8)
    axes[0, i].set_title(f'Histogram: {col}', fontsize=9)
    axes[0, i].set_xlabel(col, fontsize=8)

    # Boxplot per kelas — baris kedua
    # Berguna untuk melihat apakah fitur bisa memisahkan kelas
    groups = [df[df['target'] == j][col].values for j in range(3)]
    bp = axes[1, i].boxplot(groups, labels=['setosa', 'versicolor', 'virginica'], patch_artist=True)
    colors = ['#E6F1FB', '#E1F5EE', '#FAECE7']
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
    axes[1, i].set_title(f'Boxplot per kelas: {col}', fontsize=9)
    axes[1, i].tick_params(axis='x', rotation=15, labelsize=7)

plt.suptitle('Distribusi Fitur', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# KORELASI ANTAR FITUR
# Korelasi tinggi (mendekati 1 atau -1) berarti dua fitur
# membawa informasi yang mirip → bisa dihapus salah satunya
# Korelasi dengan target → fitur yang penting untuk prediksi
# ============================================================

# Hitung matriks korelasi (hanya kolom numerik)
korelasi = df[list(fitur_numerik) + ['target']].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(
    korelasi,
    annot=True,        # tampilkan nilai korelasi di tiap sel
    fmt='.2f',         # format 2 desimal
    cmap='coolwarm',   # warna: biru (negatif) → merah (positif)
    center=0,          # tengahkan colormap di 0
    square=True,
    linewidths=0.5
)
plt.title('Heatmap Korelasi Antar Fitur', fontsize=13)
plt.tight_layout()
plt.show()

# Interpretasi:
# petal length dan petal width sangat berkorelasi (0.96)
# Keduanya juga sangat berkorelasi dengan target
# → dua fitur terpenting untuk prediksi spesies

In [ ]:
# ============================================================
# PAIRPLOT — Visualisasi hubungan antar semua pasang fitur
# Diagonal: distribusi tiap fitur
# Off-diagonal: scatter plot antar 2 fitur, warna per kelas
# Kalau kelas bisa dipisahkan dengan jelas → fitur bagus
# ============================================================
sns.pairplot(
    df[list(fitur_numerik) + ['species']],
    hue='species',                              # warna berdasarkan kelas
    palette={'setosa': '#185FA5',
             'versicolor': '#1D9E75',
             'virginica': '#D85A30'},
    diag_kind='hist',                           # histogram di diagonal
    plot_kws={'alpha': 0.6, 's': 40}
)
plt.suptitle('Pairplot: Hubungan Antar Fitur per Kelas', y=1.01, fontsize=13)
plt.tight_layout()
plt.show()

---
## 🧹 Tahap 3: Data Preprocessing

Preprocessing adalah proses **membersihkan dan mentransformasi** data agar siap digunakan oleh algoritma ML.

Yang akan kita lakukan:
1. Simulasikan missing values (karena Iris sudah bersih)
2. Handle missing values dengan imputation
3. Feature scaling (standardisasi)

In [ ]:
# ============================================================
# SIMULASI MISSING VALUES
# Dataset Iris sudah bersih, jadi kita simulasikan missing values
# agar kamu bisa belajar cara menanganinya
# ============================================================

# Salin data asli agar tidak merusak yang original
df_dirty = df[list(fitur_numerik)].copy()

# Buat mask acak: 10% data jadi NaN
np.random.seed(42)  # agar hasil reproducible (sama setiap dijalankan)
mask = np.random.random(df_dirty.shape) < 0.10  # 10% probabilitas jadi NaN
df_dirty[mask] = np.nan

print("=== Missing Values Setelah Simulasi ===")
print(df_dirty.isnull().sum())
print(f"\nTotal NaN: {df_dirty.isnull().sum().sum()} dari {df_dirty.size} nilai")

In [ ]:
# ============================================================
# HANDLE MISSING VALUES dengan SimpleImputer
# strategy='mean'   → ganti NaN dengan rata-rata kolom
# strategy='median' → ganti NaN dengan median (lebih robust terhadap outlier)
# strategy='most_frequent' → untuk data kategorikal
# strategy='constant', fill_value=0 → ganti dengan nilai tetap
# ============================================================

# Pilih strategi: median lebih aman kalau ada outlier
imputer = SimpleImputer(strategy='median')

# fit() → imputer belajar nilai median dari data
# transform() → ganti NaN dengan nilai yang sudah dipelajari
# PENTING: nanti di workflow nyata, fit HANYA pada train set!
df_imputed = imputer.fit_transform(df_dirty)
df_imputed = pd.DataFrame(df_imputed, columns=fitur_numerik)

print("=== Missing Values Setelah Imputation ===")
print(df_imputed.isnull().sum())
print("\n✅ Semua NaN sudah terisi!")

In [ ]:
# ============================================================
# FEATURE SCALING — STANDARDISASI
# Beberapa algoritma sensitif terhadap skala fitur:
#   - SVM, KNN, Logistic Regression, MLP → HARUS di-scale
#   - Decision Tree, Random Forest, XGBoost → tidak perlu
#
# StandardScaler: x_baru = (x - mean) / std
# Hasilnya: mean=0, std=1 untuk setiap fitur
# ============================================================

# Gunakan data asli Iris (yang sudah bersih)
X = iris.data   # fitur (array numpy)
y = iris.target # target (0, 1, atau 2)

# Tampilkan range sebelum scaling
print("=== Range Fitur SEBELUM Scaling ===")
for i, nama in enumerate(fitur_numerik):
    print(f"{nama:35s}: min={X[:,i].min():.2f}, max={X[:,i].max():.2f}, mean={X[:,i].mean():.2f}")

# Buat scaler
scaler = StandardScaler()

# fit_transform: belajar mean & std, lalu transform sekaligus
X_scaled = scaler.fit_transform(X)

print("\n=== Range Fitur SETELAH Scaling ===")
for i, nama in enumerate(fitur_numerik):
    print(f"{nama:35s}: min={X_scaled[:,i].min():.2f}, max={X_scaled[:,i].max():.2f}, mean={X_scaled[:,i].mean():.2f}")

In [ ]:
# ============================================================
# ENCODING VARIABEL KATEGORIKAL
# Demonstrasi dengan data buatan karena Iris sudah numerik
# ============================================================

# --- Contoh data dengan fitur kategorikal ---
df_contoh = pd.DataFrame({
    'ukuran': ['kecil', 'besar', 'sedang', 'kecil', 'besar'],  # ordinal: ada urutan
    'kota':   ['Jakarta', 'Bandung', 'Jakarta', 'Surabaya', 'Bandung'],  # nominal: tidak ada urutan
    'harga':  [100, 300, 200, 120, 280]  # target numerik
})

print("=== Data Asli ===")
print(df_contoh)

# --- Label Encoding: untuk variabel ORDINAL (ada urutan) ---
# Bahaya kalau dipakai untuk nominal! ML akan mengira Bandung < Jakarta < Surabaya
mapping_ukuran = {'kecil': 0, 'sedang': 1, 'besar': 2}  # definisi urutan manual
df_contoh['ukuran_encoded'] = df_contoh['ukuran'].map(mapping_ukuran)

# --- One-Hot Encoding: untuk variabel NOMINAL (tidak ada urutan) ---
# Buat kolom biner terpisah untuk tiap nilai unik
# drop_first=True: hapus satu kolom untuk hindari multicollinearity
df_ohe = pd.get_dummies(df_contoh, columns=['kota'], drop_first=True)

print("\n=== Setelah Encoding ===")
print(df_ohe)

---
## ⚙️ Tahap 4: Feature Engineering

Feature engineering adalah proses **menciptakan fitur baru** atau **memilih fitur terbaik** untuk meningkatkan performa model.

In [ ]:
# ============================================================
# MEMBUAT FITUR BARU (Feature Creation)
# Kombinasi fitur yang ada bisa menghasilkan informasi baru
# yang tidak bisa ditangkap fitur aslinya secara terpisah
# ============================================================

df_feat = pd.DataFrame(iris.data, columns=[
    'sepal_length', 'sepal_width', 'petal_length', 'petal_width'
])
df_feat['target'] = iris.target

# Rasio: seberapa panjang petal relatif terhadap panjang sepal?
df_feat['petal_to_sepal_ratio'] = df_feat['petal_length'] / df_feat['sepal_length']

# Luas perkiraan: asumsikan sepal dan petal berbentuk elips
df_feat['sepal_area'] = df_feat['sepal_length'] * df_feat['sepal_width']
df_feat['petal_area'] = df_feat['petal_length'] * df_feat['petal_width']

# Selisih: seberapa jauh ukuran sepal vs petal?
df_feat['size_diff'] = df_feat['sepal_length'] - df_feat['petal_length']

print("=== Fitur Baru yang Dibuat ===")
print(df_feat[['petal_to_sepal_ratio', 'sepal_area', 'petal_area', 'size_diff']].head())

In [ ]:
# ============================================================
# FEATURE SELECTION — FILTER METHOD
# Hitung korelasi tiap fitur dengan target
# Fitur dengan korelasi tinggi = lebih informatif untuk prediksi
# ============================================================
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif

X_feat = df_feat.drop('target', axis=1).values
nama_fitur = df_feat.drop('target', axis=1).columns.tolist()

# f_classif: uji statistik F antara tiap fitur dan target
# mutual_info_classif: mengukur dependensi antar variabel (non-linear)
selector = SelectKBest(score_func=f_classif, k='all')
selector.fit(X_feat, iris.target)

# Tampilkan skor tiap fitur, diurutkan dari tertinggi
skor_fitur = pd.DataFrame({
    'Fitur': nama_fitur,
    'Skor F': selector.scores_.round(2),
    'p-value': selector.pvalues_.round(4)
}).sort_values('Skor F', ascending=False)

print("=== Skor Pentingnya Fitur (F-test) ===")
print(skor_fitur.to_string(index=False))

# Visualisasi
plt.figure(figsize=(10, 5))
plt.barh(skor_fitur['Fitur'], skor_fitur['Skor F'],
         color='#185FA5', edgecolor='white')
plt.xlabel('Skor F (lebih tinggi = lebih penting)')
plt.title('Feature Importance berdasarkan F-test')
plt.gca().invert_yaxis()  # fitur terpenting di atas
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# FEATURE IMPORTANCE dari Random Forest
# Cara lain melihat pentingnya fitur: latih Random Forest dulu,
# lalu cek berapa banyak tiap fitur digunakan untuk split
# Ini disebut 'embedded method' dalam feature selection
# ============================================================
from sklearn.ensemble import RandomForestClassifier

# Latih Random Forest sementara untuk melihat feature importance
rf_temp = RandomForestClassifier(n_estimators=100, random_state=42)
rf_temp.fit(iris.data, iris.target)

# Ambil importance dan beri nama
importance = pd.DataFrame({
    'Fitur': iris.feature_names,
    'Importance': rf_temp.feature_importances_
}).sort_values('Importance', ascending=True)

plt.figure(figsize=(8, 4))
plt.barh(importance['Fitur'], importance['Importance'],
         color=['#E6F1FB' if i < 2 else '#185FA5'
                for i in range(len(importance))])
plt.xlabel('Feature Importance (Gini)')
plt.title('Feature Importance dari Random Forest')
plt.tight_layout()
plt.show()

print("\n=== Skor Feature Importance ===")
print(importance.to_string(index=False))

---
## ✂️ Tahap 5: Train/Test Split

Membagi data menjadi **training set** (untuk melatih model) dan **test set** (untuk evaluasi akhir).

⚠️ **Aturan emas:** Test set hanya boleh dilihat SEKALI, di akhir setelah model selesai dibangun.

In [ ]:
# ============================================================
# TRAIN/TEST SPLIT
# test_size=0.2  → 20% data untuk test, 80% untuk training
# random_state=42 → hasil split sama setiap dijalankan (reproducible)
# stratify=y     → pastikan proporsi kelas sama di train dan test
#                  WAJIB untuk klasifikasi, terutama data imbalanced!
# ============================================================

X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% untuk test
    random_state=42,    # seed untuk reproducibility
    stratify=y          # jaga proporsi kelas
)

print(f"=== Hasil Pembagian Data ===")
print(f"Total data     : {len(X)} sampel")
print(f"Training set   : {len(X_train)} sampel ({len(X_train)/len(X)*100:.0f}%)")
print(f"Test set       : {len(X_test)} sampel ({len(X_test)/len(X)*100:.0f}%)")
print()

# Verifikasi stratified split: proporsi kelas harus sama
print("=== Proporsi Kelas (stratified split) ===")
print(f"{'Kelas':<12} {'Train':>8} {'Test':>8}")
print("-" * 30)
for kelas, nama in enumerate(iris.target_names):
    pct_train = (y_train == kelas).sum() / len(y_train) * 100
    pct_test  = (y_test == kelas).sum()  / len(y_test)  * 100
    print(f"{nama:<12} {pct_train:>7.1f}% {pct_test:>7.1f}%")

In [ ]:
# ============================================================
# SCALING YANG BENAR
# PENTING: fit scaler HANYA pada training data!
# Jika fit pada semua data (termasuk test), terjadi DATA LEAKAGE:
# test set 'bocor' informasinya ke proses training
# → estimasi performa menjadi terlalu optimis (tidak jujur)
# ============================================================

scaler = StandardScaler()

# fit() → pelajari mean dan std dari training data SAJA
# transform() → terapkan scaling menggunakan mean/std yang sudah dipelajari
X_train_scaled = scaler.fit_transform(X_train)  # fit + transform training
X_test_scaled  = scaler.transform(X_test)        # HANYA transform (jangan fit lagi!)

print("✅ Scaling selesai!")
print(f"Mean training (seharusnya ~0): {X_train_scaled.mean(axis=0).round(4)}")
print(f"Std  training (seharusnya ~1): {X_train_scaled.std(axis=0).round(4)}")

---
## 🤖 Tahap 6: Training & Perbandingan Banyak Algoritma

Kita akan melatih **9 algoritma klasifikasi** sekaligus dan membandingkan performanya.

In [ ]:
# ============================================================
# DEFINISIKAN SEMUA MODEL YANG AKAN DIBANDINGKAN
# Setiap model dibungkus dalam Pipeline bersama scaler-nya
# Pipeline memastikan scaling terjadi dengan benar
# di dalam cross-validation (tidak ada data leakage)
# ============================================================

models = {
    # Logistic Regression: model linear, cepat, interpretable
    # max_iter=1000 karena defaultnya kadang tidak cukup untuk konvergen
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=1000, random_state=42))
    ]),

    # Decision Tree: model berbasis aturan if-else, mudah dijelaskan
    # Tidak perlu scaling, tapi kita tetap masukkan pipeline untuk konsistensi
    'Decision Tree': Pipeline([
        ('scaler', StandardScaler()),
        ('model', DecisionTreeClassifier(random_state=42))
    ]),

    # Random Forest: ensemble dari banyak decision tree
    # n_estimators=100 → bangun 100 pohon dan ambil suara terbanyak
    'Random Forest': Pipeline([
        ('scaler', StandardScaler()),
        ('model', RandomForestClassifier(n_estimators=100, random_state=42))
    ]),

    # Gradient Boosting: membangun pohon secara bertahap, tiap pohon
    # memperbaiki kesalahan pohon sebelumnya
    'Gradient Boosting': Pipeline([
        ('scaler', StandardScaler()),
        ('model', GradientBoostingClassifier(n_estimators=100, random_state=42))
    ]),

    # XGBoost: implementasi gradient boosting yang sangat efisien
    # use_label_encoder=False dan eval_metric='mlogloss' untuk menghindari warning
    'XGBoost': Pipeline([
        ('scaler', StandardScaler()),
        ('model', XGBClassifier(n_estimators=100, random_state=42,
                                 eval_metric='mlogloss', verbosity=0))
    ]),

    # SVM: mencari hyperplane terbaik untuk memisahkan kelas
    # kernel='rbf' bisa menangani batas keputusan non-linear
    # HARUS di-scale!
    'SVM': Pipeline([
        ('scaler', StandardScaler()),
        ('model', SVC(kernel='rbf', probability=True, random_state=42))
    ]),

    # K-Nearest Neighbors: klasifikasi berdasarkan k tetangga terdekat
    # n_neighbors=5 → lihat 5 tetangga terdekat, ambil suara terbanyak
    # SANGAT sensitif terhadap skala, wajib scaling!
    'KNN': Pipeline([
        ('scaler', StandardScaler()),
        ('model', KNeighborsClassifier(n_neighbors=5))
    ]),

    # Naive Bayes: model probabilistik berdasarkan teorema Bayes
    # 'Naive' karena mengasumsikan semua fitur independen (jarang benar)
    # Sangat cepat dan efektif untuk teks
    'Naive Bayes': Pipeline([
        ('scaler', StandardScaler()),
        ('model', GaussianNB())
    ]),

    # MLP (Multi-Layer Perceptron): neural network sederhana
    # hidden_layer_sizes=(100, 50) → 2 hidden layer, 100 dan 50 neuron
    # max_iter=500 untuk memastikan konvergensi
    'Neural Network (MLP)': Pipeline([
        ('scaler', StandardScaler()),
        ('model', MLPClassifier(hidden_layer_sizes=(100, 50),
                                max_iter=500, random_state=42))
    ]),
}

print(f"✅ {len(models)} model siap dilatih dan dibandingkan!")

In [ ]:
# ============================================================
# LATIH SEMUA MODEL & EVALUASI DENGAN CROSS-VALIDATION
# Cross-validation: bagi data training menjadi k bagian (fold),
# latih k kali dengan tiap fold sebagai validasi sekali.
# Memberikan estimasi performa yang lebih stabil dan jujur
# dibanding hanya satu train/test split.
# ============================================================

hasil = []  # list untuk menyimpan hasil tiap model

print(f"{'Model':<25} {'CV Mean':>9} {'CV Std':>9} {'Test Acc':>10} {'Waktu(s)':>10}")
print("-" * 68)

for nama, pipeline in models.items():
    # Hitung waktu training
    start = time.time()

    # Cross-validation pada training data
    # cv=5 → 5-fold cross-validation
    # scoring='accuracy' → metrik yang digunakan
    cv_scores = cross_val_score(
        pipeline, X_train, y_train,
        cv=5, scoring='accuracy'
    )

    # Latih model pada seluruh training data
    pipeline.fit(X_train, y_train)

    # Prediksi pada test set
    y_pred = pipeline.predict(X_test)

    waktu = time.time() - start
    test_acc = accuracy_score(y_test, y_pred)

    # Simpan hasil
    hasil.append({
        'Model': nama,
        'CV Mean': cv_scores.mean(),
        'CV Std': cv_scores.std(),
        'Test Accuracy': test_acc,
        'Waktu (s)': waktu,
        'y_pred': y_pred  # simpan prediksi untuk analisis lebih lanjut
    })

    print(f"{nama:<25} {cv_scores.mean():>9.4f} {cv_scores.std():>9.4f} {test_acc:>10.4f} {waktu:>10.3f}")

# Urutkan berdasarkan CV Mean (lebih representatif dari test accuracy)
df_hasil = pd.DataFrame(hasil).sort_values('CV Mean', ascending=False).reset_index(drop=True)
df_hasil.index += 1

print("-" * 68)
print(f"\n🏆 Model Terbaik: {df_hasil.iloc[0]['Model']} (CV = {df_hasil.iloc[0]['CV Mean']:.4f})")

In [ ]:
# ============================================================
# VISUALISASI PERBANDINGAN MODEL
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
colors = ['#185FA5','#1D9E75','#BA7517','#D85A30','#534AB7',
          '#A32D2D','#0F6E56','#633806','#3C3489']

# --- Grafik 1: CV Accuracy dengan Error Bar ---
# Error bar menunjukkan standar deviasi → seberapa konsisten model
model_names = df_hasil['Model'].tolist()
cv_means = df_hasil['CV Mean'].tolist()
cv_stds = df_hasil['CV Std'].tolist()

bars = axes[0].bar(range(len(model_names)), cv_means,
                    yerr=cv_stds, capsize=5,
                    color=colors, edgecolor='white',
                    error_kw={'elinewidth': 1.5, 'ecolor': '#555'})
axes[0].set_xticks(range(len(model_names)))
axes[0].set_xticklabels(model_names, rotation=35, ha='right', fontsize=9)
axes[0].set_ylabel('CV Accuracy (5-fold)')
axes[0].set_title('Cross-Validation Accuracy (mean ± std)', fontsize=11)
axes[0].set_ylim(max(0, min(cv_means) - 0.1), 1.02)
axes[0].axhline(y=max(cv_means), color='red', linestyle='--',
                linewidth=1, alpha=0.5, label=f'Best: {max(cv_means):.3f}')
axes[0].legend(fontsize=9)

# Tambahkan label nilai di atas bar
for bar, val in zip(bars, cv_means):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.3f}', ha='center', va='bottom', fontsize=8)

# --- Grafik 2: Akurasi vs Waktu Training (scatter plot) ---
waktu_list = df_hasil['Waktu (s)'].tolist()
for i, (nama, acc, t) in enumerate(zip(model_names, cv_means, waktu_list)):
    axes[1].scatter(t, acc, color=colors[i], s=100, zorder=3, label=nama.split()[0])
    axes[1].annotate(nama.split()[0], (t, acc),
                     textcoords='offset points', xytext=(6, 3), fontsize=8)

axes[1].set_xlabel('Waktu Training (detik)')
axes[1].set_ylabel('CV Accuracy')
axes[1].set_title('Trade-off: Akurasi vs Kecepatan', fontsize=11)
# Model di pojok kiri atas = akurat SEKALIGUS cepat = ideal!
axes[1].text(0.05, 0.05, '← Cepat      Lambat →', transform=axes[1].transAxes,
             fontsize=8, color='gray')

plt.tight_layout()
plt.show()

---
## 📊 Tahap 7: Evaluasi Model Secara Mendalam

Accuracy saja tidak cukup. Kita perlu memahami **di mana model membuat kesalahan** dan **seberapa parah** kesalahan tersebut.

In [ ]:
# ============================================================
# CONFUSION MATRIX — untuk model terbaik
# Confusion matrix menunjukkan:
# - Baris: kelas sebenarnya (actual)
# - Kolom: kelas yang diprediksi model
# - Diagonal: prediksi BENAR
# - Off-diagonal: prediksi SALAH
# ============================================================

# Ambil model terbaik
nama_terbaik = df_hasil.iloc[0]['Model']
model_terbaik = models[nama_terbaik]
y_pred_terbaik = model_terbaik.predict(X_test)

print(f"=== Evaluasi: {nama_terbaik} ===")
print()

# Hitung confusion matrix
cm = confusion_matrix(y_test, y_pred_terbaik)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Tampilkan confusion matrix sebagai heatmap
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=iris.target_names)
disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title(f'Confusion Matrix\n{nama_terbaik}', fontsize=11)

# Confusion matrix versi persentase (normalized)
cm_norm = cm.astype(float) / cm.sum(axis=1)[:, np.newaxis]
disp_norm = ConfusionMatrixDisplay(confusion_matrix=cm_norm,
                                    display_labels=iris.target_names)
disp_norm.plot(ax=axes[1], cmap='Blues', colorbar=False)
axes[1].set_title('Confusion Matrix (Normalized)', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CLASSIFICATION REPORT
# Menampilkan precision, recall, f1-score untuk tiap kelas
#
# Precision = TP / (TP + FP)
#   "Dari semua yang diprediksi positif, berapa yang benar?"
#   Penting jika false positive mahal (misal: spam filter)
#
# Recall = TP / (TP + FN)
#   "Dari semua yang positif sebenarnya, berapa yang terdeteksi?"
#   Penting jika false negative berbahaya (misal: deteksi kanker)
#
# F1-Score = 2 * (Precision * Recall) / (Precision + Recall)
#   Harmonic mean — seimbang antara precision dan recall
# ============================================================

print("=== Classification Report ===")
print(classification_report(
    y_test,
    y_pred_terbaik,
    target_names=iris.target_names
))

In [ ]:
# ============================================================
# DIAGNOSIS OVERFITTING vs UNDERFITTING
# Bandingkan skor training dan test untuk tiap model
# - Train >> Test: OVERFITTING (model hafal, tidak generalisasi)
# - Train ≈ Test, keduanya rendah: UNDERFITTING (model terlalu sederhana)
# - Train ≈ Test, keduanya tinggi: GOOD FIT ✅
# ============================================================

print(f"{'Model':<25} {'Train Acc':>10} {'Test Acc':>10} {'Gap':>8} {'Status':>15}")
print("-" * 75)

for nama, pipeline in models.items():
    train_score = pipeline.score(X_train, y_train)  # skor pada data training
    test_score  = pipeline.score(X_test,  y_test)   # skor pada data test
    gap = train_score - test_score

    if gap > 0.10:    status = "⚠️  Overfitting"
    elif train_score < 0.80: status = "📉 Underfitting"
    else:             status = "✅ Good fit"

    print(f"{nama:<25} {train_score:>10.4f} {test_score:>10.4f} {gap:>8.4f} {status:>15}")

In [ ]:
# ============================================================
# CONTOH REGRESI — Dataset Diabetes
# Target: nilai perkembangan penyakit (angka kontinu)
# Metrik: R², MAE, RMSE
# ============================================================

# Load dataset regresi
diabetes = load_diabetes()
X_reg = diabetes.data
y_reg = diabetes.target

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

# Definisikan model regresi
models_reg = {
    'Linear Regression':  Pipeline([('sc', StandardScaler()), ('m', LinearRegression())]),
    'Ridge':              Pipeline([('sc', StandardScaler()), ('m', Ridge(alpha=1.0))]),
    'Lasso':              Pipeline([('sc', StandardScaler()), ('m', Lasso(alpha=0.1))]),
    'Decision Tree':      Pipeline([('sc', StandardScaler()), ('m', DecisionTreeRegressor(random_state=42))]),
    'Random Forest':      Pipeline([('sc', StandardScaler()), ('m', RandomForestRegressor(n_estimators=100, random_state=42))]),
    'Gradient Boosting':  Pipeline([('sc', StandardScaler()), ('m', GradientBoostingRegressor(n_estimators=100, random_state=42))]),
    'XGBoost':            Pipeline([('sc', StandardScaler()), ('m', XGBRegressor(n_estimators=100, random_state=42, verbosity=0))]),
    'SVR':                Pipeline([('sc', StandardScaler()), ('m', SVR(kernel='rbf'))]),
    'KNN Regressor':      Pipeline([('sc', StandardScaler()), ('m', KNeighborsRegressor(n_neighbors=5))]),
}

print(f"{'Model':<22} {'R² Score':>9} {'RMSE':>9} {'MAE':>9}")
print("-" * 55)

hasil_reg = []
for nama, pipeline in models_reg.items():
    pipeline.fit(X_train_r, y_train_r)
    y_pred_r = pipeline.predict(X_test_r)

    r2   = r2_score(y_test_r, y_pred_r)
    rmse = np.sqrt(mean_squared_error(y_test_r, y_pred_r))
    mae  = mean_absolute_error(y_test_r, y_pred_r)

    hasil_reg.append({'Model': nama, 'R²': r2, 'RMSE': rmse, 'MAE': mae})
    print(f"{nama:<22} {r2:>9.4f} {rmse:>9.2f} {mae:>9.2f}")

df_reg = pd.DataFrame(hasil_reg).sort_values('R²', ascending=False)
print(f"\n🏆 Terbaik: {df_reg.iloc[0]['Model']} (R² = {df_reg.iloc[0]['R²']:.4f})")

---
## 🎛️ Tahap 8: Hyperparameter Tuning

Hyperparameter adalah setelan model yang kita tentukan SEBELUM training (bukan dipelajari dari data). Tuning yang baik bisa meningkatkan performa signifikan.

Kita akan bandingkan 3 metode: **GridSearchCV**, **RandomizedSearchCV**, dan cara manual.

In [ ]:
# ============================================================
# METODE 1: GridSearchCV
# Mencoba SEMUA kombinasi parameter yang ditentukan
# Pro: exhaustive, tidak ada kombinasi yang terlewat
# Con: sangat lambat jika ruang parameter besar
# ============================================================

# Ruang parameter yang akan dicari
# Nama parameter: 'nama_step__nama_parameter'
# 'model__n_estimators' → parameter n_estimators pada step bernama 'model'
param_grid = {
    'model__n_estimators': [50, 100, 200],        # jumlah pohon
    'model__max_depth': [None, 5, 10],             # kedalaman maksimum pohon
    'model__min_samples_split': [2, 5],            # min sampel untuk split
    'model__min_samples_leaf': [1, 2],             # min sampel di daun
}
# Total kombinasi: 3 × 3 × 2 × 2 = 36 kombinasi
# Dengan cv=5: 36 × 5 = 180 model dilatih!

# Pipeline untuk tuning
rf_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier(random_state=42))
])

grid_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    cv=5,               # 5-fold cross-validation
    scoring='accuracy', # metrik yang dioptimalkan
    n_jobs=-1,          # gunakan semua CPU yang tersedia
    verbose=0           # 0=diam, 1=progress, 2=detail
)

print("Menjalankan GridSearchCV... (bisa memakan waktu)")
start = time.time()
grid_search.fit(X_train, y_train)
print(f"Selesai dalam {time.time()-start:.1f} detik")

print(f"\n=== Hasil GridSearchCV ===")
print(f"Parameter terbaik: {grid_search.best_params_}")
print(f"CV score terbaik : {grid_search.best_score_:.4f}")
print(f"Test score       : {grid_search.score(X_test, y_test):.4f}")

In [ ]:
# ============================================================
# METODE 2: RandomizedSearchCV
# Mencoba SEJUMLAH kombinasi parameter SECARA ACAK
# Pro: jauh lebih cepat, sering memberikan hasil hampir sama
# Con: tidak exhaustive, tapi dengan n_iter yang cukup, baik
# Rekomendasi: gunakan ini sebagai titik awal!
# ============================================================
from scipy.stats import randint, uniform

# Bisa menggunakan distribusi probabilitas, bukan list diskret
param_dist = {
    'model__n_estimators': randint(50, 300),     # angka acak antara 50-300
    'model__max_depth': [None, 3, 5, 7, 10, 15], # pilihan diskret
    'model__min_samples_split': randint(2, 10),   # angka acak antara 2-10
    'model__min_samples_leaf': randint(1, 5),
    'model__max_features': ['sqrt', 'log2', None]
}

random_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=param_dist,
    n_iter=30,          # coba 30 kombinasi acak (bukan semua kombinasi)
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    random_state=42,    # untuk reproducibility
    verbose=0
)

print("Menjalankan RandomizedSearchCV...")
start = time.time()
random_search.fit(X_train, y_train)
print(f"Selesai dalam {time.time()-start:.1f} detik")

print(f"\n=== Hasil RandomizedSearchCV ===")
print(f"Parameter terbaik: {random_search.best_params_}")
print(f"CV score terbaik : {random_search.best_score_:.4f}")
print(f"Test score       : {random_search.score(X_test, y_test):.4f}")

In [ ]:
# ============================================================
# LEARNING CURVE
# Melihat bagaimana performa berubah seiring penambahan data training
# Berguna untuk:
#   - Mendiagnosis overfitting/underfitting
#   - Menentukan apakah kita perlu lebih banyak data
# ============================================================
from sklearn.model_selection import learning_curve

# Hitung learning curve untuk model terbaik
train_sizes, train_scores, val_scores = learning_curve(
    models['Random Forest'],  # model yang dievaluasi
    X, y,                     # gunakan semua data
    train_sizes=np.linspace(0.1, 1.0, 10),  # dari 10% hingga 100% data
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

# Hitung mean dan std
train_mean = train_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
val_mean   = val_scores.mean(axis=1)
val_std    = val_scores.std(axis=1)

plt.figure(figsize=(9, 5))

# Plot kurva training
plt.plot(train_sizes, train_mean, 'o-', color='#185FA5',
         label='Training score', linewidth=2)
plt.fill_between(train_sizes, train_mean - train_std,
                 train_mean + train_std, alpha=0.1, color='#185FA5')

# Plot kurva validasi
plt.plot(train_sizes, val_mean, 'o-', color='#1D9E75',
         label='Validation score (CV)', linewidth=2)
plt.fill_between(train_sizes, val_mean - val_std,
                 val_mean + val_std, alpha=0.1, color='#1D9E75')

plt.xlabel('Jumlah Sampel Training')
plt.ylabel('Accuracy')
plt.title('Learning Curve — Random Forest\n'
          '(Gap besar = overfitting, keduanya rendah = underfitting)',
          fontsize=11)
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 💾 Tahap 9: Simpan, Load, dan Gunakan Model

Model yang sudah dilatih perlu disimpan agar bisa digunakan ulang tanpa melatih dari awal.

In [ ]:
# ============================================================
# SIMPAN MODEL TERBAIK
# joblib lebih efisien dari pickle untuk model scikit-learn
# karena lebih baik dalam menangani array NumPy besar
# ============================================================

# Latih model final dengan parameter terbaik dari tuning
model_final = grid_search.best_estimator_  # ambil pipeline terbaik dari GridSearch

# Simpan ke file .pkl
nama_file = 'model_iris_terbaik.pkl'
joblib.dump(model_final, nama_file)
print(f"✅ Model disimpan sebagai '{nama_file}'")

# Cek ukuran file
import os
ukuran = os.path.getsize(nama_file) / 1024
print(f"   Ukuran file: {ukuran:.1f} KB")

In [ ]:
# ============================================================
# LOAD MODEL DARI FILE
# Simulasikan penggunaan model di aplikasi nyata:
# load → beri input baru → dapatkan prediksi
# ============================================================

# Load model dari file (tidak perlu latih ulang!)
model_loaded = joblib.load(nama_file)
print("✅ Model berhasil di-load!")

# --- Simulasi prediksi untuk data baru ---
# Di dunia nyata: ini adalah input dari user/sistem
# Format: [[sepal_length, sepal_width, petal_length, petal_width]]
data_baru = np.array([
    [5.1, 3.5, 1.4, 0.2],   # kemungkinan besar setosa
    [6.7, 3.0, 5.2, 2.3],   # kemungkinan besar virginica
    [5.9, 3.0, 4.2, 1.5],   # kemungkinan besar versicolor
])

# Prediksi kelas
prediksi = model_loaded.predict(data_baru)

# Prediksi probabilitas (probability) per kelas
# Hanya bisa jika model menggunakan predict_proba (SVC perlu probability=True)
probabilitas = model_loaded.predict_proba(data_baru)

print("\n=== Hasil Prediksi untuk Data Baru ===")
print(f"{'Input':<40} {'Prediksi':<15} {'Probabilitas (setosa, versicolor, virginica)'}")
print("-" * 100)
for i, (row, pred, prob) in enumerate(zip(data_baru, prediksi, probabilitas)):
    nama_kelas = iris.target_names[pred]
    prob_str = ', '.join([f'{p:.3f}' for p in prob])
    print(f"{str(row.tolist()):<40} {nama_kelas:<15} [{prob_str}]")

In [ ]:
# ============================================================
# RINGKASAN AKHIR: PERBANDINGAN SEMUA MODEL
# ============================================================

print("=" * 60)
print("  RINGKASAN AKHIR — KLASIFIKASI IRIS")
print("=" * 60)

display_cols = ['Model', 'CV Mean', 'CV Std', 'Test Accuracy', 'Waktu (s)']
df_display = df_hasil[display_cols].copy()
df_display['CV Mean'] = df_display['CV Mean'].round(4)
df_display['CV Std']  = df_display['CV Std'].round(4)
df_display['Test Accuracy'] = df_display['Test Accuracy'].round(4)
df_display['Waktu (s)'] = df_display['Waktu (s)'].round(3)

print(df_display.to_string(index=True))

print("\n=" * 60)
print(f"  🏆 Model terbaik    : {df_hasil.iloc[0]['Model']}")
print(f"  📊 CV Accuracy      : {df_hasil.iloc[0]['CV Mean']:.4f} ± {df_hasil.iloc[0]['CV Std']:.4f}")
print(f"  🎯 Test Accuracy    : {df_hasil.iloc[0]['Test Accuracy']:.4f}")
print("=" * 60)

print("""
Langkah selanjutnya:
  1. Coba dataset yang lebih menantang (Breast Cancer, Wine, Digits)
  2. Eksperimen dengan hyperparameter berbeda
  3. Coba teknik feature engineering yang lebih kreatif
  4. Pelajari cara handle imbalanced dataset (SMOTE, class_weight)
  5. Deploy model sebagai API menggunakan FastAPI atau Flask
""")